# Benchmark de meta-learners — S/T/X/R/CausalForest

Notebook de **exploração** (sem testes). Ajusta cinco estimadores de CATE
(S-learner, T-learner, X-learner, R-learner, CausalForest) sobre o mesmo split
temporal, com o mesmo base learner (LGBM da config) e a mesma propensity fixa
onde o estimador aceita — e compara todos por Qini/AUUC, com IC de bootstrap e
teste de placebo.

## 0. Setup

Mesmo boilerplate de `notebooks/2_modeling.ipynb` — o walk-up de `ROOT` também lida com os dois níveis de `notebooks/appendix/`.

In [1]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import gc
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from causalml.inference.meta import BaseSRegressor, BaseTRegressor, BaseXRegressor, BaseRRegressor
from causalml.inference.tree import CausalRandomForestRegressor

from src import split, uplift, uplift_eval, viz
from src.uplift import (
    _design_matrix, _make_learner, fixed_propensity,
    OFFER_TYPE_COLUMN, TREATMENT_COLUMN, OUTCOME_COLUMN, GRAIN_COLUMNS,
)
from src.config import load
from src.pipeline import build_spark

pd.set_option("display.max_columns", None)
viz.setup_theme()

cfg = load()
spark = build_spark(cfg, app_name="ifood-uplift-benchmark")
processed = spark.read.parquet(str(cfg.processed_dir))
print(f"{processed.count():,} rows in the processed dataset.")

/home/caioolubini/Projects/ifood-coupons-uplift/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Failed to import duecredit due to No module named 'duecredit'
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/16 17:15:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


76,277 rows in the processed dataset.


## 1. Dado — mesmo split temporal e exclusão de informational do notebook 2

Split por `campaign_wave`, sem `informational`, JVM encerrada antes de causalml/LGBM alocarem.

In [2]:
train_sdf, holdout_sdf = split.temporal_split(processed, cfg)
split.assert_temporal_order(train_sdf, holdout_sdf)

train_sdf = split.exclude_informational(train_sdf)
holdout_sdf = split.exclude_informational(holdout_sdf)

train_df = train_sdf.toPandas()
holdout_df = holdout_sdf.toPandas()

print(f"train: {len(train_df):,} rows")
print(f"holdout: {len(holdout_df):,} rows")

26/07/16 17:15:54 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


train: 40,630 rows
holdout: 20,412 rows


In [3]:
# Stop JVM before causalml/LGBM allocate so heaps do not coexist in RAM.
spark.stop()
del processed, train_sdf, holdout_sdf
gc.collect()

25

## 2. Os cinco estimadores

`src/uplift.py` só ajusta X-learner em produção; os outros quatro são definidos aqui, sem tocar `src/`.

In [ ]:
# NOTA: a produção (`src.uplift`) só ajusta X-learner. Aqui S/T/X/R compartilham
# o MESMO base learner (`_make_learner(cfg)`, LGBM dos hiperparâmetros
# `xlearner_*`) e a MESMA propensity fixa (`fixed_propensity`) da produção —
# a comparação isola o *meta-learner*, não o base learner. LGBM tolera os
# nulos legítimos de `age`/`credit_card_limit`/`hist_*` (G8) nativamente.
#
# CausalForest não é meta-learner (árvores próprias, `fit(X, treatment, y)` sem
# `p=`) e por isso não é base-learner-matched; entra com seus próprios
# hiperparâmetros, alinhados só onde a API permite (`n_estimators`, `max_depth`,
# `random_state`). Sua árvore de decisão (via `sklearn.check_array`) **não**
# tolera NaN como o LGBM — imputação por mediana do treino é um segundo desvio
# do "mesmo dado" só para este estimador, registrado aqui e na nota final.


def _fit_meta_learner(learner_cls, df, cfg):
    models = {}
    for offer_type, group in df.groupby(OFFER_TYPE_COLUMN):
        X = _design_matrix(group)
        treatment = group[TREATMENT_COLUMN].to_numpy()
        y = group[OUTCOME_COLUMN].to_numpy()
        p = fixed_propensity(treatment)

        model = learner_cls(learner=_make_learner(cfg), control_name=0)
        model.fit(X=X, treatment=treatment, y=y, p=p)
        models[offer_type] = model
    return models


def _predict_meta_learner(models, df):
    uplift_col = pd.Series(index=df.index, dtype=float)
    for offer_type, group in df.groupby(OFFER_TYPE_COLUMN):
        model = models[offer_type]
        p = fixed_propensity(group[TREATMENT_COLUMN].to_numpy())
        te = model.predict(X=_design_matrix(group), p=p)
        uplift_col.loc[group.index] = te[:, 0] if te.ndim == 2 else te.ravel()
    return df[[*GRAIN_COLUMNS, OFFER_TYPE_COLUMN]].assign(uplift=uplift_col)


def _fit_causal_forest(df, cfg):
    models = {}
    for offer_type, group in df.groupby(OFFER_TYPE_COLUMN):
        X = _design_matrix(group)
        medianas = X.median()
        X = X.fillna(medianas).to_numpy()
        treatment = group[TREATMENT_COLUMN].to_numpy()
        y = group[OUTCOME_COLUMN].to_numpy()

        model = CausalRandomForestRegressor(
            n_estimators=cfg.xlearner_n_estimators,
            max_depth=None if cfg.xlearner_max_depth < 0 else cfg.xlearner_max_depth,
            control_name=0,
            random_state=cfg.seed,
        )
        model.fit(X=X, treatment=treatment, y=y)
        models[offer_type] = (model, medianas)
    return models


def _predict_causal_forest(models, df):
    uplift_col = pd.Series(index=df.index, dtype=float)
    for offer_type, group in df.groupby(OFFER_TYPE_COLUMN):
        model, medianas = models[offer_type]
        X = _design_matrix(group).fillna(medianas).to_numpy()
        te = model.predict(X=X)
        uplift_col.loc[group.index] = te.ravel()
    return df[[*GRAIN_COLUMNS, OFFER_TYPE_COLUMN]].assign(uplift=uplift_col)


LEARNERS = {
    "S-learner": (lambda d, c: _fit_meta_learner(BaseSRegressor, d, c), _predict_meta_learner),
    "T-learner": (lambda d, c: _fit_meta_learner(BaseTRegressor, d, c), _predict_meta_learner),
    "X-learner": (lambda d, c: _fit_meta_learner(BaseXRegressor, d, c), _predict_meta_learner),
   #"R-learner": (lambda d, c: _fit_meta_learner(BaseRRegressor, d, c), _predict_meta_learner),
   #"CausalForest": (_fit_causal_forest, _predict_causal_forest),
}

In [5]:
# Ajusta os cinco no treino e prevê no holdout — uma vez, reusado nas seções seguintes.
fitted_models = {}
uplift_scores = {}

for nome, (fit_fn, predict_fn) in LEARNERS.items():
    modelos = fit_fn(train_df, cfg)
    pred = predict_fn(modelos, holdout_df)
    fitted_models[nome] = modelos
    uplift_scores[nome] = pred["uplift"]
    print(f"{nome}: ajustado.")

S-learner: ajustado.
T-learner: ajustado.
X-learner: ajustado.
CausalForest: ajustado.


## 3. Qini/AUUC — o placar cru

`uplift_eval.qini_by_strategy` no mesmo holdout, para os cinco learners.

In [6]:
placar = uplift_eval.qini_by_strategy(
    holdout_df["converted"], holdout_df["treatment"], uplift_scores
).sort_values("qini", ascending=False).reset_index(drop=True)
placar.round(4)

,strategy,qini,auuc
0,S-learner,0.0897,0.1074
1,T-learner,0.0419,0.0499
2,CausalForest,0.0401,0.0456
3,X-learner,0.0335,0.0403


## 4. IC de bootstrap do Qini

Reamostra o holdout com reposição (`cfg.gain_curve_n_bootstrap`, `cfg.gain_curve_confidence_level`) e recomputa Qini/AUUC por réplica — sem refit, mesmo padrão de `gaincurve.gain_curves_with_ci`.

In [7]:
def _bootstrap_qini_ci(y_true, treatment, score, cfg):
    alpha = 1.0 - cfg.gain_curve_confidence_level
    rng = np.random.default_rng(cfg.seed)
    n = len(y_true)

    y_arr, t_arr, s_arr = y_true.to_numpy(), treatment.to_numpy(), score.to_numpy()
    replicas = np.empty(cfg.gain_curve_n_bootstrap)
    for i in range(cfg.gain_curve_n_bootstrap):
        idx = rng.integers(0, n, size=n)
        replicas[i] = uplift_eval.qini(pd.Series(y_arr[idx]), pd.Series(s_arr[idx]), pd.Series(t_arr[idx]))
    lo, hi = np.quantile(replicas, [alpha / 2, 1 - alpha / 2])
    return float(lo), float(hi)


bootstrap_ci = {
    nome: _bootstrap_qini_ci(holdout_df["converted"], holdout_df["treatment"], score, cfg)
    for nome, score in uplift_scores.items()
}
pd.DataFrame(
    [{"strategy": nome, "qini_lo": lo, "qini_hi": hi} for nome, (lo, hi) in bootstrap_ci.items()]
).round(4)

,strategy,qini_lo,qini_hi
0,S-learner,0.0776,0.1013
1,T-learner,0.0292,0.0551
2,X-learner,0.0187,0.0470
3,CausalForest,0.0278,0.0530


## 5. Placebo por learner

Generaliza `uplift_eval.placebo_qini_distribution`/`placebo_test` para os cinco `fit_fn`/`predict_fn` de §2. `n_permutacoes=5`, não os 20 de `cfg.placebo_n_permutations` — desvio de apêndice: 5 learners × 20 refits, com R-learner (cross-fit de 5 folds) e CausalForest (floresta inteira), é proibitivo para um Run All. Mesmo em 5, a nula ainda dá média/percentil/p-valor empírico utilizáveis.

In [8]:
N_PERMUTACOES = 5


def _placebo_distribution(fit_fn, predict_fn, train_df, holdout_df, cfg, n_permutacoes):
    scores = np.empty(n_permutacoes)
    for i in range(n_permutacoes):
        rng = np.random.default_rng(cfg.seed + i)
        placebo_train = train_df.copy()
        placebo_train["treatment"] = uplift_eval._permute_treatment_within_offer_type(placebo_train, rng)

        modelos = fit_fn(placebo_train, cfg)
        pred = predict_fn(modelos, holdout_df)
        scores[i] = uplift_eval.qini(holdout_df["converted"], pred["uplift"], holdout_df["treatment"])
    return scores


placebo_results = {}
for nome, (fit_fn, predict_fn) in LEARNERS.items():
    nula = _placebo_distribution(fit_fn, predict_fn, train_df, holdout_df, cfg, N_PERMUTACOES)
    limiar = float(np.quantile(nula, cfg.placebo_confidence_level))
    qini_real = placar.loc[placar["strategy"] == nome, "qini"].iloc[0]
    placebo_results[nome] = {
        "placebo_null_mean": float(nula.mean()),
        "placebo_p95": limiar,
        "p_value": float((nula >= qini_real).mean()),
        "passou": bool(qini_real > limiar),
    }
    print(f"{nome}: placebo concluído.")

pd.DataFrame(placebo_results).T.round(4)

S-learner: placebo concluído.
T-learner: placebo concluído.
X-learner: placebo concluído.
CausalForest: placebo concluído.


,placebo_null_mean,placebo_p95,p_value,passou
S-learner,-0.009783,0.005295,0.0,True
T-learner,-0.004301,0.004043,0.0,True
X-learner,-0.013579,0.005656,0.0,True
CausalForest,-0.006032,0.031725,0.2,True


## 6. Tabela final

Placar (§3) + IC de bootstrap (§4) + placebo (§5), uma linha por learner, ordenado por Qini.

In [9]:
ci_df = pd.DataFrame(
    [{"strategy": nome, "qini_lo": lo, "qini_hi": hi} for nome, (lo, hi) in bootstrap_ci.items()]
)
placebo_df = pd.DataFrame(placebo_results).T.reset_index(names="strategy")

tabela_final = (
    placar.merge(ci_df, on="strategy").merge(placebo_df, on="strategy")
    .sort_values("qini", ascending=False).reset_index(drop=True)
)
tabela_final = tabela_final[[
    "strategy", "qini", "qini_lo", "qini_hi", "auuc",
    "placebo_null_mean", "placebo_p95", "p_value", "passou",
]]
tabela_final.round(4)

,strategy,qini,qini_lo,qini_hi,auuc,placebo_null_mean,placebo_p95,p_value,passou
0,S-learner,0.0897,0.0776,0.1013,0.1074,-0.009783,0.005295,0.0,True
1,T-learner,0.0419,0.0292,0.0551,0.0499,-0.004301,0.004043,0.0,True
2,CausalForest,0.0401,0.0278,0.0530,0.0456,-0.006032,0.031725,0.2,True
3,X-learner,0.0335,0.0187,0.0470,0.0403,-0.013579,0.005656,0.0,True


## 7. Curvas Qini/AUUC e lucro incremental por decil

`uplift_eval.qini_curves_by_strategy` (já testado, nunca usado em notebook) — todos os learners na mesma grade, contra o diagonal aleatório. Depois, o lucro incremental marginal por decil de cada ranking, na fatoração de `gaincurve` (conversões incrementais × lucro médio por conversão tratada).

In [ ]:
curvas_qini = uplift_eval.qini_curves_by_strategy(
    holdout_df["converted"], holdout_df["treatment"], uplift_scores
)

ordem = tabela_final["strategy"].tolist()
melhor = ordem[0]
qini_score_melhor = tabela_final.set_index("strategy").loc[melhor, "qini"]
curvas_qini_ordenadas = curvas_qini.set_index("strategy").loc[ordem].reset_index()

# Todas as curvas terminam no mesmo ponto (em N=todos, a ordenação não importa
# mais) — o ranking aleatório é a reta desse ponto até a origem.
fim_qini = curvas_qini.loc[curvas_qini["strategy"] == melhor].iloc[-1]
viz.line_series(
    curvas_qini_ordenadas, x="n_treated", y="gain", group="strategy",
    title=f"{melhor} concentra mais ganho acumulado — Qini {qini_score_melhor:.3f}",
    subtitle="Ganho incremental real acumulado por nº de clientes tratados, ordenado por uplift previsto — todos os learners vs. o diagonal aleatório.",
    xlabel="clientes tratados", ylabel="ganho acumulado",
    reference=([0, fim_qini["n_treated"]], [0, fim_qini["gain"]], "random"),
)

In [ ]:
from sklift.metrics import uplift_curve

# uplift_curve tem a mesma forma de qini_curve (n_treated, gain), mas contra a
# normalização do AUUC em vez da ótima do Qini — src/uplift_eval.py só expõe o
# ponto Qini; aqui é o mesmo padrão para o outro lado do REQ-203.
partes_auuc = []
for nome, score in uplift_scores.items():
    n_treated, gain = uplift_curve(holdout_df["converted"], score, holdout_df["treatment"])
    partes_auuc.append(pd.DataFrame({"strategy": nome, "n_treated": n_treated, "gain": gain}))
curvas_auuc = pd.concat(partes_auuc, ignore_index=True)

auuc_score_melhor = tabela_final.set_index("strategy").loc[melhor, "auuc"]
curvas_auuc_ordenadas = curvas_auuc.set_index("strategy").loc[ordem].reset_index()

fim_auuc = curvas_auuc.loc[curvas_auuc["strategy"] == melhor].iloc[-1]
viz.line_series(
    curvas_auuc_ordenadas, x="n_treated", y="gain", group="strategy",
    title=f"{melhor} também lidera em AUUC — {auuc_score_melhor:.3f}",
    subtitle="Curva de ganho contra a normalização do AUUC, mesma ordenação por uplift previsto — todos os learners vs. o diagonal aleatório.",
    xlabel="clientes tratados", ylabel="ganho acumulado",
    reference=([0, fim_auuc["n_treated"]], [0, fim_auuc["gain"]], "random"),
)

In [ ]:
from src import gaincurve

# Lucro incremental por decil, na MESMA fatoração de `gaincurve` (§5 do notebook 2):
# conversões incrementais (contrafactual escalado estilo Qini sobre `converted`)
# × lucro médio por conversão tratada (`conversion_value − reward_cost`, o ticket
# líquido do custo). Marginal por decil = diferença do acumulado entre dois cortes,
# mesmo padrão de `gaincurve._marginal_conversions_by_decile`, aqui sobre `gain`.
#
# `incremental_gain_curve` aplica o envelope monótono, então o marginal por decil
# é sempre ≥ 0 por construção: um decil que destruísse lucro aparece como 0, não
# negativo. A leitura é "quanto de lucro incremental cada faixa do ranking
# acrescenta ao melhor prefixo travável", não "o lucro cru daquela faixa".
holdout_com_lucro = gaincurve.add_net_profit(holdout_df)
n_holdout = len(holdout_df)
cortes = [0] + [round(n_holdout * d / 10) for d in range(1, 11)]

partes_decil = []
for nome in ordem:
    ranking = uplift_scores[nome].sort_values(ascending=False, kind="stable").index.to_numpy()
    ordenado = holdout_com_lucro.loc[ranking].reset_index(drop=True)
    curva = gaincurve.incremental_gain_curve(np.arange(n_holdout), ordenado)

    acumulado = curva.set_index("n").loc[cortes]
    partes_decil.append(pd.DataFrame({
        "strategy": nome,
        "decil": np.arange(1, 11),
        "lucro_incremental": np.diff(acumulado["gain"].to_numpy()),
        "conversoes_incrementais": np.diff(acumulado["conversions"].to_numpy()),
    }))

lucro_por_decil = pd.concat(partes_decil, ignore_index=True)
lucro_por_decil.pivot(index="decil", columns="strategy", values="lucro_incremental").round(2)

In [ ]:
lucro_decil_wide = (
    lucro_por_decil.pivot(index="decil", columns="strategy", values="lucro_incremental")
    [ordem].reset_index()
)
lucro_d1_melhor = lucro_decil_wide.loc[lucro_decil_wide["decil"] == 1, melhor].iloc[0]
viz.grouped_bars(
    lucro_decil_wide, category="decil", series=ordem,
    title=f"{melhor} concentra R$ {viz.format_pt_br(lucro_d1_melhor)} de lucro incremental no D1",
    subtitle="Lucro líquido incremental marginal por decil do ranking (D1 = topo) — conversões incrementais × lucro médio por conversão tratada.",
    xlabel="decil do ranking", ylabel="lucro incremental (R$)",
)

## 8. S-learner — mergulho na distribuição de μ1 − μ0

O S-learner ajusta um único modelo com o tratamento como feature e prevê duas vezes (`treatment=0`/`1`, `predict(..., return_components=True)`); τ = μ1 − μ0 é a diferença entre as duas previsões, não dois modelos separados como no X-learner. Média, mediana, desvio e histograma no holdout, por `offer_type`.

In [12]:
def _s_learner_components(models, df):
    mu0 = pd.Series(index=df.index, dtype=float)
    mu1 = pd.Series(index=df.index, dtype=float)
    for offer_type, group in df.groupby(OFFER_TYPE_COLUMN):
        model = models[offer_type]
        X = _design_matrix(group)
        p = fixed_propensity(group[TREATMENT_COLUMN].to_numpy())
        _, yhat_cs, yhat_ts = model.predict(X=X, p=p, return_components=True)
        arm = model.t_groups[0]
        mu0.loc[group.index] = yhat_cs[arm]
        mu1.loc[group.index] = yhat_ts[arm]
    return df[[*GRAIN_COLUMNS, OFFER_TYPE_COLUMN]].assign(mu0=mu0, mu1=mu1, tau=mu1 - mu0)


s_learner_components = _s_learner_components(fitted_models["S-learner"], holdout_df)

s_learner_stats = s_learner_components["tau"].agg(["mean", "median", "std"]).to_frame("mu1_menos_mu0").T
s_learner_stats.round(4)

,mean,median,std
mu1_menos_mu0,0.058,0.0534,0.0709


In [13]:
media_tau = s_learner_stats.loc["mu1_menos_mu0", "mean"]
mediana_tau = s_learner_stats.loc["mu1_menos_mu0", "median"]
viz.histogram(
    s_learner_components, x="tau",
    title=f"S-learner: τ médio {media_tau:.3f}, mediana {mediana_tau:.3f}",
    subtitle="Distribuição de μ1 − μ0 prevista no holdout, uma previsão por linha.",
    xlabel="τ = μ1 − μ0",
    markers=[(media_tau, "média"), (mediana_tau, "mediana")],
)

### 8.1 S-learner vs. prior de conversão

O S-learner ajusta um único modelo com o tratamento como mais uma feature — se a árvore raramente usa `treatment` para splitar, τ colapsa em algo próximo de uma função da propensão a converter. Correlação de Pearson (linear) e Spearman (ordenação, a que importa para Qini) entre τ e o P(converte) do `ConversionModel` (mesmo LGBM baseline do blend de produção), ajustado no mesmo treino.

In [ ]:
from src.models import ConversionModel

p_convert = ConversionModel.from_config(cfg).fit(train_df).predict_proba(holdout_df)

tau_s = s_learner_components["tau"]
correlacoes = pd.DataFrame([
    {"metodo": "pearson", "correlacao": tau_s.corr(p_convert, method="pearson")},
    {"metodo": "spearman", "correlacao": tau_s.corr(p_convert, method="spearman")},
])
correlacoes.round(4)

In [ ]:
# 20.412 pontos num scatter viram uma mancha — o τ médio por decil de P(converte)
# mostra a forma da relação (monótona? achatada?) que as duas correlações resumem.
tau_por_decil_conv = (
    pd.DataFrame({"tau": tau_s, "p_convert": p_convert})
    .assign(decil=lambda d: pd.qcut(d["p_convert"], 10, labels=False, duplicates="drop") + 1)
    .groupby("decil", as_index=False)
    .agg(tau_medio=("tau", "mean"))
)

spearman = correlacoes.set_index("metodo").loc["spearman", "correlacao"]
viz.line_series(
    tau_por_decil_conv, x="decil", y="tau_medio",
    title=f"τ do S-learner vs. propensão a converter — Spearman {spearman:.3f}",
    subtitle="τ médio previsto por decil de P(converte) do prior de conversão, no holdout (D10 = maior propensão).",
    xlabel="decil de P(converte)", ylabel="τ médio",
    mode="lines+markers", end_labels=False,
)

### 8.2 Importância causal do S-learner, com intervalos

Mesma API avançada do CausalML que `uplift.causal_importance` usa para o X-learner (`get_importance` com `model_tau_feature` sobre o τ̂ estimado, `method='permutation'`, reconciliada entre os `offer_type` por média ponderada pelo nº de linhas) — `get_importance` vive em `BaseLearner`, então vale igual para o S-learner. Mede o que dirige o **efeito**, não o outcome.

O intervalo vem de repetir a permutação com `random_state` diferente (`N_REPETICOES_IMPORTANCIA`) e tomar o percentil `cfg.gain_curve_confidence_level` das réplicas: é a incerteza **da permutação e do split interno** do meta-modelo, não a incerteza amostral do holdout (essa exigiria bootstrap com refit do S-learner inteiro por réplica, proibitivo aqui). Lê-se "quão estável é esse ranking de importância", não "quanto ele variaria em outra amostra de clientes".

In [ ]:
from src.uplift import _XLEARNER_FEATURES

N_REPETICOES_IMPORTANCIA = 10


def _importancia_causal_replica(models, df, cfg, random_state):
    """Uma réplica de `uplift.causal_importance`, com o `random_state` da permutação.

    Mesma álgebra do X-learner em `src/uplift.py` (piso em 0, média ponderada pelo
    nº de linhas entre os `offer_type`, normalização para somar 1) — só o
    `random_state` muda entre réplicas.
    """
    total = pd.Series(0.0, index=_XLEARNER_FEATURES)
    n_total = 0

    for offer_type, group in df.groupby(OFFER_TYPE_COLUMN):
        model = models[offer_type]
        X = _design_matrix(group)
        p = fixed_propensity(group[TREATMENT_COLUMN].to_numpy())
        tau = model.predict(X=X, p=p)

        imp = model.get_importance(
            X=X, tau=tau,
            model_tau_feature=_make_learner(cfg),
            features=np.array(_XLEARNER_FEATURES),
            method="permutation",
            random_state=random_state,
        )
        (group_imp,) = imp.values()
        group_imp = group_imp.reindex(_XLEARNER_FEATURES).fillna(0.0).clip(lower=0.0)
        total = total + group_imp * len(group)
        n_total += len(group)

    media = total / n_total
    return media / media.sum() if media.sum() > 0 else media


replicas_imp = pd.DataFrame([
    _importancia_causal_replica(fitted_models["S-learner"], holdout_df, cfg, cfg.seed + i)
    for i in range(N_REPETICOES_IMPORTANCIA)
])

alpha_imp = 1.0 - cfg.gain_curve_confidence_level
importancia_s = (
    pd.DataFrame({
        "importancia": replicas_imp.mean(),
        "importancia_lo": replicas_imp.quantile(alpha_imp / 2),
        "importancia_hi": replicas_imp.quantile(1 - alpha_imp / 2),
    })
    .sort_values("importancia", ascending=False)
    .rename_axis("feature")
    .reset_index()
)
importancia_s.round(4)

In [ ]:
TOP_N_IMPORTANCIA = 12

top_imp = importancia_s.head(TOP_N_IMPORTANCIA)
lider = top_imp.iloc[0]

# `horizontal_bars` não tem barra de erro (nenhuma figura de src/ precisou até
# aqui); `labels=False` é o caso que ele já documenta para importância
# normalizada, e o IC entra por `update_traces` sem estilo ad hoc — a cor e o
# tema seguem do tema único.
fig_imp = viz.horizontal_bars(
    top_imp, category="feature", value="importancia",
    title=f"{lider['feature']} dirige o efeito no S-learner — {lider['importancia']:.1%} da importância causal",
    subtitle=f"Importância causal por permutação sobre o τ̂ estimado, normalizada; IC {cfg.gain_curve_confidence_level:.0%} de {N_REPETICOES_IMPORTANCIA} repetições da permutação. Top {TOP_N_IMPORTANCIA}.",
    xlabel="participação na importância causal",
    labels=False,
)
ordenado_imp = top_imp.sort_values("importancia", ascending=True)
fig_imp.update_traces(
    error_x=dict(
        type="data", symmetric=False,
        array=(ordenado_imp["importancia_hi"] - ordenado_imp["importancia"]).to_numpy(),
        arrayminus=(ordenado_imp["importancia"] - ordenado_imp["importancia_lo"]).to_numpy(),
        color=viz.ink()[1], thickness=1.5, width=5,
    ),
    hovertemplate="%{y}<br>%{x:.3f}<br>IC [%{customdata[0]:.3f}, %{customdata[1]:.3f}]<extra></extra>",
    customdata=np.column_stack([ordenado_imp["importancia_lo"], ordenado_imp["importancia_hi"]]),
)
fig_imp

### 8.3 O quanto o S-learner usa o `treatment`?

`treatment` **não aparece** na §8.2 e não é omissão: `get_importance` mede um meta-modelo sobre `X → τ̂(X)`, e τ̂ = μ₁(x) − μ₀(x) é função **só de x** — o tratamento já foi marginalizado ao avaliar o modelo interno duas vezes (`t=0`/`t=1`) e subtrair. Não existe "importância do treatment para τ̂".

Onde o `treatment` existe é **dentro** do S-learner: o CausalML o prepende como coluna 0 (`concat_treatment_col` no fit, `prepend_column` no predict), então o modelo interno é `[treatment | X] → y`. Aqui vale a pergunta que é a patologia clássica do S-learner: **a árvore de fato splita no tratamento, ou o ignora?** Se ignora, τ colapsa para perto de constante e o modelo não é de uplift — é um preditor de conversão disfarçado (o que a §8.1 mede pelo outro lado).

Importância por permutação (`sklearn.inspection.permutation_importance`, mesma grandeza da §8.2 — queda de acurácia ao embaralhar a coluna), `cfg.gain_curve_n_bootstrap` não se aplica: o IC vem das `n_repeats` repetições da permutação. Isto é importância **preditiva de `y`**, não do efeito — é o único eixo em que `treatment` tem significado.

In [ ]:
from sklearn.inspection import permutation_importance

# O modelo interno do S-learner é `[treatment | X] → y`: o CausalML prepende o
# tratamento como coluna 0 e renomeia tudo para 0..n, então o índice de nomes é
# ["treatment"] + _XLEARNER_FEATURES, nessa ordem. `_design_matrix` já entrega as
# categóricas como códigos inteiros, e os nulos legítimos do G8 sobrevivem ao
# hstack (o LGBM os trata nativamente).
NOMES_INTERNOS = ["treatment"] + list(_XLEARNER_FEATURES)


def _importancia_interna_replicas(models, df, cfg):
    """Importância por permutação no modelo interno do S-learner, por offer_type.

    Devolve a matriz (repetição × feature) ponderada pelo nº de linhas e
    normalizada por réplica — mesma álgebra de agregação da §8.2, mas sobre o
    modelo interno, que é o único lugar onde `treatment` existe.
    """
    total = None
    n_total = 0

    for offer_type, group in df.groupby(OFFER_TYPE_COLUMN):
        model = models[offer_type]
        X = _design_matrix(group)
        w = (group[TREATMENT_COLUMN].to_numpy() == 1).astype(float)
        X_interno = pd.DataFrame(np.hstack([w.reshape(-1, 1), X.to_numpy()]), index=X.index)
        y = group[OUTCOME_COLUMN].to_numpy()

        inner = model.models[model.t_groups[0]]
        r = permutation_importance(
            inner, X_interno, y,
            n_repeats=N_REPETICOES_IMPORTANCIA,
            random_state=cfg.seed,
            scoring="neg_mean_squared_error",
        )
        # (feature × repetição) → (repetição × feature), piso em 0 como na §8.2.
        imp = np.clip(r.importances.T, 0.0, None)
        total = imp * len(group) if total is None else total + imp * len(group)
        n_total += len(group)

    media = total / n_total
    somas = media.sum(axis=1, keepdims=True)
    return pd.DataFrame(
        np.divide(media, somas, out=np.zeros_like(media), where=somas > 0),
        columns=NOMES_INTERNOS,
    )


replicas_internas = _importancia_interna_replicas(fitted_models["S-learner"], holdout_df, cfg)

importancia_interna = (
    pd.DataFrame({
        "importancia": replicas_internas.mean(),
        "importancia_lo": replicas_internas.quantile(alpha_imp / 2),
        "importancia_hi": replicas_internas.quantile(1 - alpha_imp / 2),
    })
    .sort_values("importancia", ascending=False)
    .rename_axis("feature")
    .reset_index()
)

posicao_treatment = importancia_interna.index[importancia_interna["feature"] == "treatment"][0] + 1
peso_treatment = importancia_interna.iloc[posicao_treatment - 1]
print(
    f"treatment: {peso_treatment['importancia']:.1%} da importância preditiva do modelo interno "
    f"(IC [{peso_treatment['importancia_lo']:.1%}, {peso_treatment['importancia_hi']:.1%}]), "
    f"posição {posicao_treatment} de {len(importancia_interna)}."
)
importancia_interna.round(4)

In [ ]:
# `treatment` sempre no gráfico, mesmo se não estiver no top-N — é a feature da
# pergunta. Destacado por cor de estado + "(tratamento)" no rótulo: a cor nunca
# carrega o significado sozinha (docstring de src/viz.py).
top_interna = importancia_interna.head(TOP_N_IMPORTANCIA)
if "treatment" not in top_interna["feature"].to_numpy():
    top_interna = pd.concat([top_interna, importancia_interna.query("feature == 'treatment'")])

top_interna = top_interna.assign(
    rotulo=lambda d: np.where(d["feature"] == "treatment", "treatment (tratamento)", d["feature"])
)
ordenada_interna = top_interna.sort_values("importancia", ascending=True)

fig_interna = viz.horizontal_bars(
    top_interna, category="rotulo", value="importancia",
    title=f"O S-learner dá {peso_treatment['importancia']:.1%} da importância preditiva ao tratamento",
    subtitle=f"Importância por permutação no modelo interno `[treatment | X] → converted`, normalizada; IC {cfg.gain_curve_confidence_level:.0%} de {N_REPETICOES_IMPORTANCIA} repetições. Preditiva de y, não do efeito.",
    xlabel="participação na importância preditiva",
    labels=False,
)
fig_interna.update_traces(
    marker_color=[
        viz.STATUS["critical"] if r == "treatment (tratamento)" else viz.palette()[0]
        for r in ordenada_interna["rotulo"]
    ],
    error_x=dict(
        type="data", symmetric=False,
        array=(ordenada_interna["importancia_hi"] - ordenada_interna["importancia"]).to_numpy(),
        arrayminus=(ordenada_interna["importancia"] - ordenada_interna["importancia_lo"]).to_numpy(),
        color=viz.ink()[1], thickness=1.5, width=5,
    ),
    hovertemplate="%{y}<br>%{x:.3f}<br>IC [%{customdata[0]:.3f}, %{customdata[1]:.3f}]<extra></extra>",
    customdata=np.column_stack([ordenada_interna["importancia_lo"], ordenada_interna["importancia_hi"]]),
)
fig_interna